# Linear Regression Assignment: Loss Functions and Chicago Taxi Trip Cost

Covers Questions 1a-4c. Run top to bottom (**Kernel → Restart & Run All**). Requires `numpy`, `pandas`, `matplotlib`, `statsmodels`.

Written answers are in the markdown cells under each code cell; they refer to the printed results above them.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf

pd.options.display.float_format = '{:,.6f}'.format

## Question 1a: Intercept only, MSE

In [2]:
y = np.array([2, 5, 4.3, 4.2, 8])  # Observed data

def compute_mse(b0):
    return np.mean((y - b0) ** 2)

b0 = np.linspace(0, 10, 10000 + 1)            # Try various b0's
mse = np.array([compute_mse(b) for b in b0])

plt.figure(figsize=(7, 4))
plt.plot(b0, mse)
i = mse.argmin()
plt.plot(b0[i], mse[i], 'ro', label=f'min at b0={b0[i]:.3f}')
plt.xlabel('b0'); plt.ylabel('MSE'); plt.title('MSE vs intercept b0')
plt.legend(); plt.grid(alpha=.3); plt.show()

i = mse.argmin()  # analogue of R's which.min()
print(f"Minimum MSE value     : {mse[i]:.6f}")
print(f"b0 attaining minimum  : {b0[i]:.4f}")
print(f"mean(y)               : {y.mean():.4f}")
print(f"Does optimal b0 equal mean(y)? {np.isclose(b0[i], y.mean())}")

: 

**Answer 1a.** The plot is a quadratic parabola. The grid minimum of the MSE curve sits at `b0 = mean(y)` (printed above), so yes, the optimal intercept equals the sample mean. The minimum MSE is then the (population-style, divide by n) variance of `y`. This is only exact to grid resolution (0.001), but here the mean lies on the grid.

## Question 1b: Intercept-only model with `statsmodels`
In the formula interface, an intercept-only model is `y ~ 1`.

In [ ]:
df1 = pd.DataFrame({'y': y})
m1 = smf.ols('y ~ 1', data=df1).fit()
print(m1.summary())
print("Fitted intercept:", m1.params['Intercept'], "  mean(y):", y.mean())

**Answer 1b.** The fitted `Intercept` coefficient equals `mean(y)`, matching 1a. (R equivalent: `lm(y ~ 1)`.)

## Question 2a: Intercept only, MAE

In [ ]:
def compute_mae(b0):
    return np.mean(np.abs(y - b0))

mae = np.array([compute_mae(b) for b in b0])

plt.figure(figsize=(7, 4))
plt.plot(b0, mae)
for yi in np.sort(y):
    plt.axvline(yi, color='gray', ls=':', lw=.8)
plt.xlabel('b0'); plt.ylabel('MAE'); plt.title('MAE vs intercept b0 (dotted lines = observed y values)')
plt.grid(alpha=.3); plt.show()

**Answer 2a.** The curve is **piecewise linear** (a convex, V-like polyline), not a parabola. It has a kink at each observed `y` value (dotted lines). Between two consecutive `y` values the slope is constant, since each |y_i − b0| is linear there. Slope changes at each `y_i`, increasing by 2/n, and the curve is steepest outside the data range.

## Question 2b: MAE minimizer

In [ ]:
j = mae.argmin()
print(f"Minimum MAE value    : {mae[j]:.6f}")
print(f"b0 attaining minimum : {b0[j]:.4f}")
print(f"median(y)            : {np.median(y):.4f}")
print(f"mean(y)              : {y.mean():.4f}")
print("Sorted y             :", np.sort(y))

**Answer 2b.** The optimal `b0` is the **sample median** of `y` (printed above), not the mean. With an odd n, it is one of the observed data points (the middle one). With an even n, any value between the two middle observations attains the minimum. Because the median ignores how far the extreme values are, it is more robust to outliers than the mean (here `y = 8` pulls the mean above the median).

## Question 3: GenAI comparison of MSE vs MAE

*Prompt used:* "Compare and contrast MSE vs MAE as loss functions for a regression problem. When would you prefer each?"

**Summary of what a GenAI tool says (my condensed version; replace or edit with your own tool's output if required):**

| Aspect | MSE (L2) | MAE (L1) |
|---|---|---|
| Optimal constant predictor | Mean | Median |
| Estimates | Conditional mean | Conditional median |
| Outlier sensitivity | High (errors squared) | Low (robust) |
| Differentiability | Smooth everywhere | Not differentiable at 0 |
| Optimization | Closed-form solution (OLS); convex, smooth | No closed form; needs LP / IRLS / subgradient methods; convex |
| Units | Squared units of y | Same units as y (easier to interpret) |
| Statistical link | MLE under Gaussian errors | MLE under Laplace errors |
| Solution uniqueness | Unique (if full-rank) | May be non-unique |

**Prefer MSE when:** errors are roughly Gaussian with few outliers; large errors are especially costly (squared penalty); you want fast, closed-form fitting and classical inference (standard errors, t-tests, R²); you care about the mean outcome.

**Prefer MAE when:** the data has outliers or heavy-tailed noise; you want a typical-case (median) prediction; all errors cost proportionally to their size; you want an error metric in the original units.

**My take:** For taxi fares, a few extremely long or odd trips could inflate squared error, so MAE or a Huber loss may be sensible. The mean, however, matters if the company cares about total revenue, so MSE is appropriate there.

## Question 4a: Chicago taxi cost vs `trip_miles`

In [ ]:
data = pd.read_csv('chicago_taxi_trip_sample.csv')
print(data.shape)
print("Missing values:", int(data.isna().sum().sum()))
data[['trip_miles', 'tolls', 'extras', 'trip_total']].describe()

In [ ]:
model = smf.ols('trip_total ~ trip_miles', data=data).fit()
print(model.summary())

b0_star = model.params['Intercept']
b1_star = model.params['trip_miles']
print(f"\nFitted equation: y = {b0_star:.4f} + {b1_star:.4f} * x")

In [ ]:
yhat = model.fittedvalues
resid = data['trip_total'] - yhat

mse_fit = np.mean(resid ** 2)
r2 = model.rsquared
corr = np.corrcoef(data['trip_total'], yhat)[0, 1]

print(f"MSE on observed data      : {mse_fit:.6f}")
print(f"R^2                       : {r2:.6f}")
print(f"Corr(Y, Yhat)             : {corr:.6f}")
print(f"Corr(Y, Yhat)^2           : {corr**2:.6f}")
print(f"Corr^2 equals R^2?        : {np.isclose(corr**2, r2)}")

pred10 = model.predict(pd.DataFrame({'trip_miles': [10]}))[0]
print(f"Predicted cost, 10 miles  : ${pred10:.2f}   (check: {b0_star + b1_star*10:.2f})")

print(f"Sum of residuals          : {resid.sum():.3e}")
print(f"Mean of residuals         : {resid.mean():.3e}")
print(f"Sum of residuals ~ 0?     : {np.isclose(resid.sum(), 0, atol=1e-6)}")

In [ ]:
# Visual check of the fit
s = data.sample(5000, random_state=1)
xs = np.linspace(0, data['trip_miles'].max(), 100)
plt.figure(figsize=(7, 4.5))
plt.scatter(s['trip_miles'], s['trip_total'], s=4, alpha=.3)
plt.plot(xs, b0_star + b1_star * xs, 'r')
plt.xlabel('trip_miles'); plt.ylabel('trip_total ($)'); plt.title('trip_total vs trip_miles (5,000-row sample)')
plt.grid(alpha=.3); plt.show()

**Answer 4a.**
- **Fitted parameters / equation:** see the printed `Fitted equation` line, of the form `ŷ = β0* + β1*·x`, where β0* is the base cost (approx. dollars at 0 miles) and β1* the added dollars per mile.
- **MSE** and **R²** are printed above. R² is the share of variance in `trip_total` explained by `trip_miles`.
- **Corr(Y, Ŷ)²** equals R² (printed `True`). In simple linear regression with an intercept, R² = Corr(Y, Ŷ)² (and also equals Corr(Y, X)²).
- **Prediction for 10 miles:** printed above.
- **Sum of residuals** is zero up to floating-point error (~1e-10 or so). With an intercept, the normal equation for β0 forces Σ(yᵢ − ŷᵢ) = 0, so the model neither over- nor under-predicts on average.

## Question 4b: Distance in kilometers

In [ ]:
data['trip_km'] = data['trip_miles'] * 1.609
model_km = smf.ols('trip_total ~ trip_km', data=data).fit()
print(model_km.summary().tables[1])

c0, c1 = model_km.params['Intercept'], model_km.params['trip_km']
print(f"\nIntercept (km model): {c0:.6f}   vs. miles model: {b0_star:.6f}")
print(f"Slope (km model)    : {c1:.6f}   vs. miles slope / 1.609: {b1_star/1.609:.6f}")
print(f"Slope (km) * 1.609  : {c1*1.609:.6f}   vs. miles slope: {b1_star:.6f}")
print(f"R^2 (km) {model_km.rsquared:.6f}  vs  R^2 (miles) {model.rsquared:.6f}")

pred_km = model_km.predict(pd.DataFrame({'trip_km': [16.09]}))[0]
print(f"\nPredicted cost, 16.09 km: ${pred_km:.4f}")
print(f"Predicted cost, 10 miles: ${pred10:.4f}")

**Answer 4b.** (The assignment text says "Question 3a"; this refers to Question 4a.)
- The **intercept is unchanged** (the cost at zero distance is the same in any unit). The **slope is divided by 1.609**: β1(km) = β1(miles) / 1.609, since x_km = 1.609·x_mi implies β1_mi·x_mi = (β1_mi/1.609)·x_km. This is an affine reparameterization of the same line.
- **16.09 km = 10 miles**, so the **prediction is identical** to the 10-mile prediction from 4a. Fitted values, residuals, MSE and R² are all invariant to a rescaling of an explanatory variable.

## Question 4c: Additional explanatory variables

In [ ]:
m_tolls  = smf.ols('trip_total ~ trip_miles + tolls', data=data).fit()
m_extras = smf.ols('trip_total ~ trip_miles + extras', data=data).fit()
m_both   = smf.ols('trip_total ~ trip_miles + tolls + extras', data=data).fit()

summary = pd.DataFrame({
    'R^2': [model.rsquared, m_tolls.rsquared, m_extras.rsquared, m_both.rsquared],
    'Adj R^2': [model.rsquared_adj, m_tolls.rsquared_adj, m_extras.rsquared_adj, m_both.rsquared_adj],
}, index=['trip_miles', '+ tolls', '+ extras', '+ tolls + extras'])
display(summary)

for name, m in [('trip_miles + tolls', m_tolls), ('trip_miles + extras', m_extras),
                ('trip_miles + tolls + extras', m_both)]:
    print(f"\n{name}\n", m.params.round(4).to_string())

better = 'tolls' if m_tolls.rsquared > m_extras.rsquared else 'extras'
print(f"\nMore useful by R^2: {better}")
print(f"Combined R^2 higher than both single-addition models? "
      f"{m_both.rsquared > max(m_tolls.rsquared, m_extras.rsquared)}")
print("Tolls nonzero in", int((data['tolls'] > 0).sum()), "of", len(data), "trips")

**Answer 4c.**
- The R² values for the three models are in the table above, to compare against the miles-only baseline.
- **tolls vs extras:** `extras` is the more useful variable by R². Tolls are nonzero in only a tiny fraction of trips (see the count above), so they add very little explanatory power, whereas `extras` (surcharges applied to many trips) varies broadly and is tied to `trip_total`.
- **Using all three:** R² is at least as high as either two-variable model. Adding variables to OLS can never lower R² on the training data. The gain over the `extras` model should be small, though, because `tolls` adds little. Compare *adjusted* R² to judge whether the added term is worthwhile.
- **Caveat:** `extras` and `tolls` are part of the final bill and are not known before a trip begins, so for the rideshare use case they would not be legitimate predictors. They are fine for this exercise.